# Data Preprocessing Pipelines
Data preprocessing pipelines for the MRNet, KneeMRI, CAI2R, and FastMRI datasets. Uploaded the processed files to HuggingFace.

This notebook, including explanations and code structure, was generated with the assistance of a large language model (Gemini).

ChatGPT was used to assist in loading the datasets and uploading to HuggingFace,

In [ ]:
!pip install numpy opencv-python nibabel pandas huggingface_hub datasets h5py opencv-python

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import numpy as np
import cv2

def get_middle_slice(scan):
    if len(scan.shape) == 3:
        # MRNet: (slices, H, W)
        return scan[len(scan)//2]
    return scan

def resize(img):
    return cv2.resize(img, (224, 224))

def normalize(img):
    img = img - np.min(img)
    img = img / (np.max(img) + 1e-8)
    return img

def process_scan(scan):
    img = get_middle_slice(scan)
    # Convert complex numbers to their magnitude (absolute value) before resizing
    if np.iscomplexobj(img):
        img = np.abs(img)
    img = resize(img)
    img = normalize(img)
    return img

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## MRNet Dataset Processing

This section of the notebook focuses on processing the MRNet dataset. It involves:

*   **Data Loading and Feature Definition:** Importing necessary libraries, defining the base paths for raw data and processed output, and setting up the schema for the Hugging Face dataset (sagittal, coronal, axial views as `Array3D` and a `label` as `Value`).
*   **Split Processing:** A function `process_split` is defined to read labels from CSV files, load corresponding `.npy` MRI scans, and store them in memory buffers. These buffers are then converted into `datasets.Dataset` objects and saved to disk in `CHUNK_SIZE` shards to manage memory.
*   **Dataset Aggregation and Upload:** The saved shards are loaded and concatenated to form complete `train_ds` and `val_ds` datasets. Finally, these datasets are combined into a `DatasetDict` and pushed to the Hugging Face Hub, making them accessible for model training or sharing.


In [ ]:
import os
import numpy as np
import pandas as pd
from datasets import Dataset, Features, Array3D, Value

BASE_PATH = "/content/drive/MyDrive/All Documents/Auburn/Semesters/Spring 2026/COMP 5630/ACL Project/raw/MRNet"   # CHANGE
SAVE_PATH = "/content/drive/MyDrive/All Documents/Auburn/Semesters/Spring 2026/COMP 5630/ACL Project/raw/mrnet_hf"
CHUNK_SIZE = 50

In [ ]:
features = Features({
    "sagittal": Array3D((32,224,224), "float16"),
    "coronal":  Array3D((32,224,224), "float16"),
    "axial":    Array3D((32,224,224), "float16"),
    "label":    Value("int64"),
})

In [ ]:
def process_split(split, csv_file):
    labels = pd.read_csv(os.path.join(BASE_PATH, csv_file), header=None)

    os.makedirs(f"{SAVE_PATH}/{split}", exist_ok=True)

    buffer = []
    shard_id = 0

    for i, row in labels.iterrows():
        case_id = str(row[0]).zfill(4)
        label = int(row[1])

        sample = {
            "sagittal": np.load(f"{BASE_PATH}/{split}/sagittal/{case_id}.npy").astype("float16"),
            "coronal":  np.load(f"{BASE_PATH}/{split}/coronal/{case_id}.npy").astype("float16"),
            "axial":    np.load(f"{BASE_PATH}/{split}/axial/{case_id}.npy").astype("float16"),
            "label": label
        }

        buffer.append(sample)

        if len(buffer) == CHUNK_SIZE:
            ds = Dataset.from_list(buffer, features=features)
            ds.save_to_disk(f"{SAVE_PATH}/{split}/shard_{shard_id}")
            buffer = []
            shard_id += 1

    if buffer:
        ds = Dataset.from_list(buffer, features=features)
        ds.save_to_disk(f"{SAVE_PATH}/{split}/shard_{shard_id}")

In [ ]:
process_split("train", "train-acl.csv")
process_split("valid", "valid-acl.csv")

Saving the dataset (0/2 shards):   0%|          | 0/50 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/50 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/50 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/50 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/50 [00:00<?, ? examples/s]

...

Saving the dataset (0/2 shards):   0%|          | 0/50 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/30 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/50 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/50 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/20 [00:00<?, ? examples/s]

In [ ]:
from datasets import load_from_disk, concatenate_datasets

def load_split(split):
    shards = []
    path = f"{SAVE_PATH}/{split}"

    for folder in sorted(os.listdir(path)):
        shards.append(load_from_disk(os.path.join(path, folder)))

    return concatenate_datasets(shards)

train_ds = load_split("train")
val_ds   = load_split("valid")

In [1]:
from datasets import DatasetDict

dataset = DatasetDict({
    "train": train_ds,
    "validation": val_ds
})

dataset.push_to_hub("elizabethacasey/mrnet-acl", max_shard_size="500MB")

NameError: name 'train_ds' is not defined

## KneeMRI Dataset Processing

This section handles the processing of the KneeMRI dataset. The key steps include:

*   **Configuration:** Setting up the folder path for the raw KneeMRI data and initializing variables for batch processing.
*   **Hugging Face Repository Setup:** Ensuring a Hugging Face dataset repository (`elizabethacasey/acl-mri-dataset`) exists to store the processed data.
*   **Iterative Processing:** The notebook iterates through subfolders and `.pck` (pickle) files within the KneeMRI directory. It loads the MRI scans, applies the `process_scan` function (defined earlier for slice selection, resizing, and intensity normalization), and accumulates the processed images into batches.
*   **Batch Upload to Hugging Face:** Once a batch is full, the processed images are saved as a compressed `.npz` file locally, then uploaded to the specified Hugging Face repository, and finally the local `.npz` file is deleted to conserve disk space. Any remaining images are uploaded in a final partial batch.


In [ ]:
import os
from huggingface_hub import upload_file

folder = "/content/drive/MyDrive/All Documents/Auburn/Semesters/Spring 2026/COMP 5630/ACL Project/raw/KneeMRI100"

X_batch = []
batch_size = 20
batch_num = 0

In [ ]:
import pickle
import os
import numpy as np
from huggingface_hub import create_repo, upload_file

# Create the Hugging Face repository if it doesn't exist
create_repo(repo_id="elizabethacasey/acl-mri-dataset", repo_type="dataset", exist_ok=True)

processed_files_count = 0

# Iterate through the subfolders (vol01, vol02, etc.)
for subfolder_name in os.listdir(folder):
    subfolder_path = os.path.join(folder, subfolder_name)

    # Check if it's a directory and its name starts with 'vol'
    if os.path.isdir(subfolder_path) and subfolder_name.startswith('vol'):
        for file in os.listdir(subfolder_path):
            if file.endswith(".pck"):
                path = os.path.join(subfolder_path, file)

                try:
                    with open(path, "rb") as f:
                        data = pickle.load(f)
                except Exception as e:
                    print(f"Error loading pickle file {path}: {e}")
                    continue

                # handle both cases where data is a dict or a direct scan
                scan = None
                if isinstance(data, dict):
                    for key in data:
                        if isinstance(data[key], np.ndarray) and data[key].ndim == 3:
                            scan = data[key]
                            break
                    if scan is None:
                        print(f"Warning: No 3D numpy array found in dict for file {file}")
                        continue
                elif isinstance(data, np.ndarray) and data.ndim == 3:
                    scan = data
                else:
                    print(f"Warning: Unexpected data format in file {file}. Skipping.")
                    continue

                img = process_scan(scan)
                X_batch.append(img)
                processed_files_count += 1

                if len(X_batch) == batch_size:
                    filename = f"knee100_batch_{batch_num}.npz"
                    np.savez_compressed(filename, X=np.array(X_batch))

                    upload_file(
                        path_or_fileobj=filename,
                        path_in_repo=f"data/{filename}",
                        repo_id="elizabethacasey/acl-mri-dataset",
                        repo_type="dataset"
                    )

                    os.remove(filename)
                    X_batch = []
                    batch_num += 1

# Upload any remaining files that didn't form a full batch
if len(X_batch) > 0:
    filename = f"knee100_batch_{batch_num}.npz"
    np.savez_compressed(filename, X=np.array(X_batch))

    upload_file(
        path_or_fileobj=filename,
        path_in_repo=f"data/{filename}",
        repo_id="elizabethacasey/acl-mri-dataset",
        repo_type="dataset"
    )

    os.remove(filename)
    X_batch = [] # Clear for next potential run
    batch_num += 1

print(f"Finished processing {processed_files_count} KneeMRI files.")
print(f"Final batch_num: {batch_num}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  knee100_batch_0.npz         :  26%|##5       |  524kB / 2.04MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

  knee100_batch_35.npz        :  77%|#######7  | 1.57MB / 2.03MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  knee100_batch_36.npz        :  98%|#########7| 1.57MB / 1.60MB            

Finished processing 736 KneeMRI files.
Final batch_num: 37


## CAI2R Dataset Processing

This section is dedicated to processing the CAI2R dataset, which typically comes in `.mat` (MATLAB) file format:

*   **Configuration and Repository:** Defines the `cai2r_folder` path and ensures the Hugging Face dataset repository (`elizabethacasey/acl-mri-dataset`) is ready for uploads.
*   **File Iteration and Loading:** It walks through the CAI2R directory, identifies `.mat` files, and attempts to load them using `scipy.io.loadmat`. It includes error handling for MATLAB v7.3 files that require `h5py`.
*   **Data Extraction and Preprocessing:** For each loaded `.mat` file, it looks for a 'rawdata' key containing a 3D NumPy array. If found, the `process_scan` function (same as used for KneeMRI) is applied for image standardization (slice selection, resizing, and normalization).
*   **Batching and Upload:** Similar to KneeMRI, processed images are collected into batches, saved as compressed `.npz` files, uploaded to the Hugging Face Hub, and then removed locally.


In [ ]:
import scipy.io
import os
import numpy as np
from huggingface_hub import upload_file, create_repo

# Define the folder path for cai2r data
cr2i_folder = "/content/drive/MyDrive/All Documents/Auburn/Semesters/Spring 2026/COMP 5630/ACL Project/raw/cai2r"

# Create the Hugging Face repository if it doesn't exist
create_repo(repo_id="elizabethacasey/acl-mri-dataset", repo_type="dataset", exist_ok=True)

X_batch_cr2i, y_batch_cr2i = [], [] # y_batch_cr2i is currently unused as no labels were identified
batch_size_cr2i = 20 # You can adjust this batch size
batch_num_cr2i = 0
processed_files_count_cr2i = 0

# Iterate through subfolders in the cr2i_folder
for root, _, files in os.walk(cr2i_folder):
    for file in files:
        if file.endswith('.mat'):
            file_path = os.path.join(root, file)

            try:
                mat_contents = scipy.io.loadmat(file_path)
            except NotImplementedError:
                print(f"Skipping {file_path}: MATLAB v7.3 files require h5py. Please install h5py and use h5py.File to load.")
                continue
            except Exception as e:
                print(f"Error loading .mat file {file_path}: {e}")
                continue

            scan = None
            # Check for the 'rawdata' key and if it's a 3D numpy array
            if 'rawdata' in mat_contents and isinstance(mat_contents['rawdata'], np.ndarray) and mat_contents['rawdata'].ndim == 3:
                scan = mat_contents['rawdata']
            else:
                print(f"Warning: 'rawdata' not found or not a 3D array in {file_path}. Skipping.")
                continue

            # Process the scan using the previously defined function
            img = process_scan(scan)
            X_batch_cr2i.append(img)
            processed_files_count_cr2i += 1

            if len(X_batch_cr2i) == batch_size_cr2i:
                filename = f"cai2r_batch_{batch_num_cr2i}.npz"
                np.savez_compressed(filename, X=np.array(X_batch_cr2i))

                upload_file(
                    path_or_fileobj=filename,
                    path_in_repo=f"data/{filename}",
                    repo_id="elizabethacasey/acl-mri-dataset",
                    repo_type="dataset"
                )

                os.remove(filename)
                X_batch_cr2i = []
                batch_num_cr2i += 1
                print(f"Uploaded cai2r_batch_{batch_num_cr2i-1}.npz")

# Upload any remaining files that didn't form a full batch
if len(X_batch_cr2i) > 0:
    filename = f"cai2r_batch_{batch_num_cr2i}.npz"
    np.savez_compressed(filename, X=np.array(X_batch_cr2i))

    upload_file(
        path_or_fileobj=filename,
        path_in_repo=f"data/{filename}",
        repo_id="elizabethacasey/acl-mri-dataset",
        repo_type="dataset"
    )

    os.remove(filename)
    X_batch_cr2i = [] # Clear for next potential run
    batch_num_cr2i += 1
    print(f"Uploaded final cai2r_batch_{batch_num_cr2i-1}.npz")

print(f"\nFinished processing {processed_files_count_cr2i} CAI2R files.")
print(f"Final batch_num for CAI2R: {batch_num_cr2i}")

Skipping /content/drive/MyDrive/All Documents/Auburn/Semesters/Spring 2026/COMP 5630/ACL Project/raw/cai2r/1/espirit36.mat: MATLAB v7.3 files require h5py. Please install h5py and use h5py.File to load.
Skipping /content/drive/MyDrive/All Documents/Auburn/Semesters/Spring 2026/COMP 5630/ACL Project/raw/cai2r/1/espirit32.mat: MATLAB v7.3 files require h5py. Please install h5py and use h5py.File to load.
Skipping /content/drive/MyDrive/All Documents/Auburn/Semesters/Spring 2026/COMP 5630/ACL Project/raw/cai2r/1/espirit34.mat: MATLAB v7.3 files require h5py. Please install h5py and use h5py.File to load.
Skipping /content/drive/MyDrive/All Documents/Auburn/Semesters/Spring 2026/COMP 5630/ACL Project/raw/cai2r/1/espirit38.mat: MATLAB v7.3 files require h5py. Please install h5py and use h5py.File to load.
Skipping /content/drive/MyDrive/All Documents/Auburn/Semesters/Spring 2026/COMP 5630/ACL Project/raw/cai2r/1/espirit35.mat: MATLAB v7.3 files require h5py. Please install h5py and use h5py

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  cai2r_batch_0.npz           :   9%|9         |  524kB / 5.77MB            

Uploaded cai2r_batch_0.npz
Skipping /content/drive/MyDrive/All Documents/Auburn/Semesters/Spring 2026/COMP 5630/ACL Project/raw/cai2r/1/espirit29.mat: MATLAB v7.3 files require h5py. Please install h5py and use h5py.File to load.
Skipping /content/drive/MyDrive/All Documents/Auburn/Semesters/Spring 2026/COMP 5630/ACL Project/raw/cai2r/1/espirit22.mat: MATLAB v7.3 files require h5py. Please install h5py and use h5py.File to load.
Skipping /content/drive/MyDrive/All Documents/Auburn/Semesters/Spring 2026/COMP 5630/ACL Project/raw/cai2r/1/espirit28.mat: MATLAB v7.3 files require h5py. Please install h5py and use h5py.File to load.
Skipping /content/drive/MyDrive/All Documents/Auburn/Semesters/Spring 2026/COMP 5630/ACL Project/raw/cai2r/1/espirit21.mat: MATLAB v7.3 files require h5py. Please install h5py and use h5py.File to load.
Skipping /content/drive/MyDrive/All Documents/Auburn/Semesters/Spring 2026/COMP 5630/ACL Project/raw/cai2r/1/espirit27.mat: MATLAB v7.3 files require h5py. Pleas

...

Uploaded cai2r_batch_35.npz
Skipping /content/drive/MyDrive/All Documents/Auburn/Semesters/Spring 2026/COMP 5630/ACL Project/raw/cai2r/9/espirit10.mat: MATLAB v7.3 files require h5py. Please install h5py and use h5py.File to load.
Skipping /content/drive/MyDrive/All Documents/Auburn/Semesters/Spring 2026/COMP 5630/ACL Project/raw/cai2r/9/espirit13.mat: MATLAB v7.3 files require h5py. Please install h5py and use h5py.File to load.
Skipping /content/drive/MyDrive/All Documents/Auburn/Semesters/Spring 2026/COMP 5630/ACL Project/raw/cai2r/9/espirit12.mat: MATLAB v7.3 files require h5py. Please install h5py and use h5py.File to load.
Skipping /content/drive/MyDrive/All Documents/Auburn/Semesters/Spring 2026/COMP 5630/ACL Project/raw/cai2r/9/espirit2.mat: MATLAB v7.3 files require h5py. Please install h5py and use h5py.File to load.
Skipping /content/drive/MyDrive/All Documents/Auburn/Semesters/Spring 2026/COMP 5630/ACL Project/raw/cai2r/9/espirit16.mat: MATLAB v7.3 files require h5py. Pleas

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  cai2r_batch_36.npz          :  91%|######### | 2.10MB / 2.31MB            

Uploaded final cai2r_batch_36.npz

Finished processing 728 CAI2R files.
Final batch_num for CAI2R: 37


## FastMRI Dataset Processing

This section outlines the pipeline for processing the FastMRI dataset, which typically involves large tar archives of HDF5 files:

*   **Imports and Configuration:** Imports specific libraries for FastMRI (`h5py`, `tqdm`, `urllib.request`) and defines extensive configuration parameters including the `TARBALL_URL`, processing `SPLIT`, `MAX_FILES` per session, local and Drive storage paths, and Hugging Face repository details (`HF_REPO_ID`, `HF_TOKEN`, `UPLOAD_MODE`).
*   **Utility Functions:** Provides helper functions for:
    *   `load_progress` and `save_progress`: To track successful, failed, and uploaded files for resumable processing.
    *   `rss_reconstruction`: To convert k-space data (from HDF5) into image space using Root-Sum-of-Squares.
    *   `h5_to_npz`: To extract k-space from HDF5, perform RSS reconstruction, apply slice trimming and min-max normalization, and save as compressed `.npz` files with metadata.
    *   `tmp_free_gb` and `drive_free_gb`: To monitor disk space.
*   **Hugging Face Setup:** Logs into Hugging Face and prepares the target repository, including a helper `upload_npz_to_hf` function.
*   **Main Processing Loop (Streaming):** This is the core logic. It streams the large `.tar.xz` archive, extracts `.h5` files one by one to temporary storage (`/tmp`), processes them using `h5_to_npz`, saves the `.npz` output to Google Drive, and optionally uploads them to Hugging Face in streaming mode (if `UPLOAD_MODE` is 'streaming'). Progress is tracked and saved throughout.
*   **Batch Upload (for 'drive' mode or leftovers):** A separate cell handles batch uploads of `.npz` files that are stored on Google Drive but not yet on Hugging Face. It stages files in batches, commits them as single uploads to the Hub, and updates the upload progress.


In [ ]:
import os, json, shutil, time
from pathlib import Path
from datetime import datetime

import h5py
import numpy as np
from tqdm.notebook import tqdm
from huggingface_hub import HfApi, login as hf_login

In [ ]:
FASTMRI_BASE_URL = "https://nam11.safelinks.protection.outlook.com/?url=https%3A%2F%2Ffastmri-dataset.s3.amazonaws.com%2Fv2.0%2Fknee_singlecoil_train.tar.xz%3FAWSAccessKeyId%3DAKIAJM2LEZ67Y2JL3KRA%26Signature%3DMACQQ9I7NGQaVcrCBGD97wMUSLo%253D%26Expires%3D1779387997&data=05%7C02%7Ceac0123%40auburn.edu%7C9141f169640e4c206d5708de70ad960b%7Cccb6deedbd294b388979d72780f62d3b%7C0%7C0%7C639072088310034879%7CUnknown%7CTWFpbGZsb3d8eyJFbXB0eU1hcGkiOnRydWUsIlYiOiIwLjAuMDAwMCIsIlAiOiJXaW4zMiIsIkFOIjoiTWFpbCIsIldUIjoyfQ%3D%3D%7C40000%7C%7C%7C&sdata=OV9doCmfabZ0nc%2BIP3%2ByikarwZxTy8C7ezssGzHsgcw%3D&reserved=0"  # e.g. "https://fastmri-dataset.s3.amazonaws.com/"
FASTMRI_TOKEN    = ""  # if NYU issued you an auth token (leave blank if not)

# Which split to process
SPLIT = "train"

# ── File selection ────────────────────────────────────────────────────────────
# How many .h5 files to process in this run (set to None to process all)
MAX_FILES = None

# ── Storage ──────────────────────────────────────────────────────────────────
DRIVE_ROOT   = "/content/drive/MyDrive/fastmri_npz"  # output folder on Drive
TMP_DIR      = Path("/tmp/fastmri")                  # scratch space in Colab
PROGRESS_FILE = Path(DRIVE_ROOT) / "progress.json"  # tracks completed files

# ── HuggingFace ───────────────────────────────────────────────────────────────
HF_REPO_ID   = "elizabethacasey/fastmri-knee-singlecoil-rss"  # create this repo first
HF_TOKEN     = userdata.get('HF_TOKEN')   # or set env var HUGGINGFACE_TOKEN

# Upload mode: "streaming"  → upload each .npz right after creating it (deletes local copy)
#              "drive"      → keep all .npz on Drive, upload in a separate step
UPLOAD_MODE  = "drive"   # start with 'drive' to be safe; switch to 'streaming' once confirmed

# ── Processing ───────────────────────────────────────────────────────────────
# Max slices to keep per volume (None = keep all)
# Useful to cap storage if volumes have 30–50 slices each
MAX_SLICES_PER_VOLUME = None

Config OK


In [ ]:
TARBALL_URL = "https://nam11.safelinks.protection.outlook.com/?url=https%3A%2F%2Ffastmri-dataset.s3.amazonaws.com%2Fv2.0%2Fknee_singlecoil_train.tar.xz%3FAWSAccessKeyId%3DAKIAJM2LEZ67Y2JL3KRA%26Signature%3DMACQQ9I7NGQaVcrCBGD97wMUSLo%253D%26Expires%3D1779387997&data=05%7C02%7Ceac0123%40auburn.edu%7C9141f169640e4c206d5708de70ad960b%7Cccb6deedbd294b388979d72780f62d3b%7C0%7C0%7C639072088310034879%7CUnknown%7CTWFpbGZsb3d8eyJFbXB0eU1hcGkiOnRydWUsIlYiOiIwLjAuMDAwMCIsIlAiOiJXaW4zMiIsIkFOIjoiTWFpbCIsIldUIjoyfQ%3D%3D%7C40000%7C%7C%7C&sdata=OV9doCmfabZ0nc%2BIP3%2ByikarwZxTy8C7ezssGzHsgcw%3D&reserved=0"  # e.g. "https://fastmri-dataset.s3.amazonaws.com/knee_singlecoil_train.tar.gz?AWSAccessKeyId=..."

# How many .h5 files to process per session (safe to increase once confirmed working)
MAX_FILES = 10

Tarball URL set (583 chars). Ready to stream.


In [ ]:
import urllib.request


# ── Progress tracking ────────────────────────────────────────────────────────

def load_progress() -> dict:
    """Load set of already-processed filenames from Drive."""
    if PROGRESS_FILE.exists():
        with open(PROGRESS_FILE) as f:
            return json.load(f)
    return {"completed": [], "failed": [], "uploaded": []}


def save_progress(progress: dict):
    with open(PROGRESS_FILE, "w") as f:
        json.dump(progress, f, indent=2)


# ── Download ─────────────────────────────────────────────────────────────────

def download_h5(url: str, dest: Path) -> bool:
    """Download a single .h5 file to dest. Returns True on success."""
    try:
        headers = {}
        if FASTMRI_TOKEN:
            headers["Authorization"] = f"Bearer {FASTMRI_TOKEN}"
        req = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(req, timeout=120) as response, open(dest, "wb") as out:
            shutil.copyfileobj(response, out)
        return True
    except Exception as e:
        print(f"  ✗ Download failed: {e}")
        return False


# ── RSS Reconstruction ───────────────────────────────────────────────────────

def rss_reconstruction(kspace: np.ndarray) -> np.ndarray:
    """
    Root-Sum-of-Squares reconstruction from k-space.
    Input:  kspace  shape (slices, coils, height, width) complex64   [multicoil]
            OR      shape (slices, height, width) complex64           [singlecoil]
    Output: images  shape (slices, height, width) float32
    """
    # Inverse FFT to image space
    images = np.fft.ifftshift(
        np.fft.ifft2(
            np.fft.ifftshift(kspace, axes=(-2, -1)),
            axes=(-2, -1)
        ),
        axes=(-2, -1)
    )
    # |magnitude|
    images = np.abs(images)
    if images.ndim == 4:           # multicoil: RSS across coil dim
        images = np.sqrt((images ** 2).sum(axis=1))
    return images.astype(np.float32)


# ── .h5 → .npz ───────────────────────────────────────────────────────────────

def h5_to_npz(h5_path: Path, npz_path: Path) -> dict:
    """
    Extract k-space from .h5, reconstruct RSS images, save as .npz.
    Returns a metadata dict.
    """
    with h5py.File(h5_path, "r") as f:
        kspace = f["kspace"][:]          # full volume loaded into RAM
        attrs  = dict(f.attrs)           # acquisition metadata

    # Optional: trim slices
    if MAX_SLICES_PER_VOLUME is not None:
        kspace = kspace[:MAX_SLICES_PER_VOLUME]

    images = rss_reconstruction(kspace)

    # Normalise to [0, 1] per volume  (optional but recommended)
    vmax = images.max()
    if vmax > 0:
        images = images / vmax

    meta = {
        "filename":   h5_path.stem,
        "num_slices": images.shape[0],
        "height":     images.shape[1],
        "width":      images.shape[2],
        "acquisition": str(attrs.get("acquisition", "")),
        "num_low_frequency": int(attrs.get("num_low_frequency", 0)),
    }

    np.savez_compressed(npz_path, images=images, **{k: str(v) for k, v in meta.items()})
    return meta


# ── Storage check ────────────────────────────────────────────────────────────

def tmp_free_gb() -> float:
    stat = shutil.disk_usage("/tmp")
    return stat.free / 1e9


def drive_free_gb() -> float:
    stat = shutil.disk_usage("/content/drive")
    return stat.free / 1e9


print("Utilities loaded ✓")
print(f"  /tmp free:   {tmp_free_gb():.1f} GB")
print(f"  Drive free:  {drive_free_gb():.1f} GB")

Utilities loaded ✓
  /tmp free:   92.7 GB
  Drive free:  5187.1 GB


In [ ]:
# Login — token can also be set as Colab secret (Secrets tab → HUGGINGFACE_TOKEN)
import os
token = HF_TOKEN or os.environ.get("HUGGINGFACE_TOKEN", "")
if token:
    hf_login(token=token, add_to_git_credential=False)
    api = HfApi()
    # Create repo if it doesn't exist
    try:
        api.create_repo(repo_id=HF_REPO_ID, repo_type="dataset", exist_ok=True)
        print(f"HF repo ready: https://huggingface.co/datasets/{HF_REPO_ID}")
    except Exception as e:
        print(f"Could not create repo (may already exist): {e}")
else:
    print("⚠ No HF token set — upload steps will be skipped")
    api = None


def upload_npz_to_hf(npz_path: Path, remote_folder: str = "data") -> bool:
    """Upload a single .npz file to the HF dataset repo."""
    if api is None:
        return False
    try:
        api.upload_file(
            path_or_fileobj=str(npz_path),
            path_in_repo=f"{remote_folder}/{npz_path.name}",
            repo_id=HF_REPO_ID,
            repo_type="dataset",
        )
        return True
    except Exception as e:
        print(f"  ✗ HF upload failed: {e}")
        return False

HF repo ready: https://huggingface.co/datasets/elizabethacasey/fastmri-knee-singlecoil-rss


In [ ]:
import io
import tarfile
import urllib.request
import time
from pathlib import Path

# Re-load progress in case you're resuming
progress     = load_progress()
completed_set = set(progress["completed"])
failed_set    = set(progress["failed"])
uploaded_set  = set(progress["uploaded"])

print(f"Already completed: {len(completed_set)} | Failed: {len(failed_set)}")
print(f"Opening remote stream... (may take 10–20s to connect)")

MIN_TMP_FREE_GB   = 2.0
MIN_DRIVE_FREE_GB = 1.0

processed_this_run = 0
skipped_this_run   = 0

# urllib gives us a raw byte stream from the remote .tar.gz
req = urllib.request.Request(TARBALL_URL)
with urllib.request.urlopen(req, timeout=60) as remote_stream:
    # tarfile can read a gzip stream directly — no temp file needed
    with tarfile.open(fileobj=remote_stream, mode="r|xz") as tar:
        for member in tar:
            # Only process .h5 files
            if not member.name.endswith(".h5"):
                continue

            stem = Path(member.name).stem

            # Skip if already done
            if stem in completed_set:
                skipped_this_run += 1
                continue

            # Stop once we hit the session limit
            if MAX_FILES is not None and processed_this_run >= MAX_FILES:
                print(f"\nReached MAX_FILES={MAX_FILES} for this session. Re-run to continue.")
                break

            # Storage guard
            if tmp_free_gb() < MIN_TMP_FREE_GB:
                print(f"\n⚠ /tmp only {tmp_free_gb():.1f} GB free — stopping.")
                break
            if drive_free_gb() < MIN_DRIVE_FREE_GB:
                print(f"\n⚠ Drive only {drive_free_gb():.1f} GB free — stopping.")
                break

            print(f"\n[{stem}] Extracting from stream...")
            t0 = time.time()

            h5_path  = TMP_DIR / f"{stem}.h5"
            npz_path = Path(DRIVE_ROOT) / f"{stem}.npz"

            # Extract this single member to /tmp
            try:
                f_in = tar.extractfile(member)
                if f_in is None:
                    print(f"  ✗ Could not read member — skipping")
                    continue
                with open(h5_path, "wb") as f_out:
                    shutil.copyfileobj(f_in, f_out)
            except Exception as e:
                print(f"  ✗ Extraction failed: {e}")
                failed_set.add(stem)
                progress["failed"] = list(failed_set)
                save_progress(progress)
                continue

            h5_mb = h5_path.stat().st_size / 1e6
            print(f"  Extracted {h5_mb:.0f} MB in {time.time()-t0:.1f}s")

            # Process → .npz
            try:
                meta = h5_to_npz(h5_path, npz_path)
                npz_mb = npz_path.stat().st_size / 1e6
                print(f"  Saved .npz  {npz_mb:.0f} MB  "
                      f"({meta['num_slices']} slices, {meta['height']}×{meta['width']})")
            except Exception as e:
                print(f"  ✗ Processing failed: {e}")
                failed_set.add(stem)
                progress["failed"] = list(failed_set)
                save_progress(progress)
                h5_path.unlink(missing_ok=True)
                continue

            # Delete .h5 immediately
            h5_path.unlink()

            # Upload if streaming mode
            if UPLOAD_MODE == "streaming" and api is not None:
                print(f"  Uploading to HuggingFace...")
                if upload_npz_to_hf(npz_path):
                    npz_path.unlink()
                    uploaded_set.add(stem)
                    progress["uploaded"] = list(uploaded_set)
                    print(f"  ✓ Uploaded and removed local copy")

            # Mark complete
            completed_set.add(stem)
            progress["completed"] = list(completed_set)
            save_progress(progress)
            processed_this_run += 1

print("\n" + "="*50)
print(f"Session complete.")
print(f"  Processed this run : {processed_this_run}")
print(f"  Skipped (done)     : {skipped_this_run}")
print(f"  Failed             : {len(failed_set)}")
print(f"  Drive free         : {drive_free_gb():.1f} GB")

Already completed: 0 | Failed: 0
Opening remote stream... (may take 10–20s to connect)


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive'

In [ ]:
from huggingface_hub import HfApi
import shutil
from pathlib import Path

# How many files to commit per batch.
# 100 is safe — well under the 128/hour commit limit since it's 1 commit per batch.
# Increase to 500+ once you've confirmed it works.
UPLOAD_BATCH_SIZE = 100

progress     = load_progress()
uploaded_set = set(progress["uploaded"])

npz_files = sorted(Path(DRIVE_ROOT).glob("*.npz"))
to_upload  = [p for p in npz_files if p.stem not in uploaded_set]

print(f"Files on Drive   : {len(npz_files)}")
print(f"Already uploaded : {len(uploaded_set)}")
print(f"To upload        : {len(to_upload)}")

if not api:
    print("⚠ No HF token — skipping")
else:
    # Stage files into a flat temp folder, then upload the whole folder at once.
    # upload_folder() creates ONE commit regardless of how many files are inside.
    stage_dir = Path("/tmp/hf_stage")

    for batch_start in range(0, len(to_upload), UPLOAD_BATCH_SIZE):
        batch = to_upload[batch_start : batch_start + UPLOAD_BATCH_SIZE]
        batch_end = batch_start + len(batch)
        print(f"\nBatch {batch_start+1}–{batch_end} of {len(to_upload)}")

        # Clear and rebuild staging dir
        if stage_dir.exists():
            shutil.rmtree(stage_dir)
        stage_dir.mkdir()

        for p in batch:
            shutil.copy2(p, stage_dir / p.name)

        print(f"  Staged {len(batch)} files — uploading as single commit...")
        try:
            api.upload_folder(
                folder_path=str(stage_dir),
                path_in_repo="data",
                repo_id=HF_REPO_ID,
                repo_type="dataset",
                commit_message=f"Add files {batch_start+1}–{batch_end}",
            )
            # Mark all files in this batch as uploaded
            for p in batch:
                uploaded_set.add(p.stem)
            progress["uploaded"] = list(uploaded_set)
            save_progress(progress)
            print(f"  ✓ Committed {len(batch)} files ({len(uploaded_set)} total uploaded)")

        except Exception as e:
            print(f"  ✗ Batch failed: {e}")
            print("  Files are still on Drive — re-run this cell to retry.")
            break

    # Clean up staging dir
    if stage_dir.exists():
        shutil.rmtree(stage_dir)

    print(f"\nDone. {len(uploaded_set)} files on HuggingFace.")
    print(f"View: https://huggingface.co/datasets/{HF_REPO_ID}")

Files on Drive   : 973
Already uploaded : 386
To upload        : 587

Batch 1–100 of 587
  Staged 100 files — uploading as single commit...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  .../hf_stage/file1000833.npz:  76%|#######5  | 23.9MB / 31.5MB            

  .../hf_stage/file1000829.npz: 100%|##########| 29.4MB / 29.4MB            

...

  .../hf_stage/file1002146.npz: 100%|##########| 33.8MB / 33.8MB            

  .../hf_stage/file1002088.npz: 100%|##########| 27.7MB / 27.7MB            

  .../hf_stage/file1002153.npz:  85%|########4 | 23.9MB / 28.2MB            

  .../hf_stage/file1002154.npz:  50%|#####     | 16.0MB / 32.0MB            

  ✓ Committed 87 files (973 total uploaded)

Done. 973 files on HuggingFace.
View: https://huggingface.co/datasets/elizabethacasey/fastmri-knee-singlecoil-rss
